# RAG on Raw Data (No Entity Resolution)

This notebook builds a RAG chatbot directly on the **raw** Open Sanctions and Open Ownership datasets — no Senzing entity resolution involved. The goal is to show what RAG looks like *before* entity resolution so we can compare it with the entity-resolved version in notebook 06.

**Key differences from notebook 06:**
- Vectorizes the **original source records** (340 records), not resolved entities (196)
- Creates human-readable text from each JSON record instead of vectorizing raw JSON
- Supports both **Anthropic** and **OpenAI** as the LLM backend

In [1]:
import json
import os

import anthropic
import lancedb
import openai
import pyarrow as pa
from sentence_transformers import SentenceTransformer

print("All imports successful")

All imports successful


## Clear LanceDB

Completely remove all existing LanceDB data so we start fresh with only the raw source records.

In [2]:
LANCEDB_PATH = "/workspace/lancedb_data"


def destroy_lancedb(path: str) -> None:
    """Completely clear all LanceDB data by dropping every table."""
    db = lancedb.connect(path)
    table_names = db.table_names()
    if table_names:
        for name in table_names:
            db.drop_table(name)
            print(f"Dropped table: {name}")
        print(f"Cleared {len(table_names)} table(s) from LanceDB at: {path}")
    else:
        print(f"No tables found in LanceDB at: {path}")


destroy_lancedb(LANCEDB_PATH)

Dropped table: raw_records
Cleared 1 table(s) from LanceDB at: /workspace/lancedb_data


/tmp/ipykernel_18819/3972754603.py:7: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  table_names = db.table_names()


## Load Raw Data

Load the original Open Sanctions and Open Ownership JSON files (JSONL format — one JSON object per line).

In [3]:
DATA_DIR = "/workspace/data"


def load_jsonl(filepath: str) -> list[dict]:
    """Load a JSONL file (one JSON object per line)."""
    records = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


sanctions_records = load_jsonl(os.path.join(DATA_DIR, "open-sanctions.json"))
ownership_records = load_jsonl(os.path.join(DATA_DIR, "open-ownership.json"))

print(f"Open Sanctions records: {len(sanctions_records)}")
print(f"Open Ownership records: {len(ownership_records)}")
print(f"Total raw records: {len(sanctions_records) + len(ownership_records)}")

Open Sanctions records: 24
Open Ownership records: 316
Total raw records: 340


## Build Meaningful Text from Raw Records

Instead of vectorizing the raw JSON, we create a human-readable text description for each record. This gives the embedding model meaningful semantic content to work with.

In [4]:
def get_name(record: dict) -> str:
    """Extract the primary name from a record."""
    # Open Sanctions: names are in NAMES list
    for name_entry in record.get("NAMES", []):
        if name_entry.get("NAME_FULL"):
            return name_entry["NAME_FULL"]
        if name_entry.get("NAME_ORG"):
            return name_entry["NAME_ORG"]
        if name_entry.get("PRIMARY_NAME_ORG"):
            return name_entry["PRIMARY_NAME_ORG"]
        if name_entry.get("PRIMARY_NAME_FULL"):
            return name_entry["PRIMARY_NAME_FULL"]
    # Open Ownership: sometimes name is a top-level field
    if record.get("PRIMARY_NAME_FULL"):
        return record["PRIMARY_NAME_FULL"]
    return "Unknown"


def record_to_text(record: dict) -> str:
    """Convert a raw JSON record into a meaningful human-readable string."""
    parts = []

    name = get_name(record)
    record_type = record.get("RECORD_TYPE", "UNKNOWN")
    source = record.get("DATA_SOURCE", "UNKNOWN")

    parts.append(f"{name} is a {record_type.lower()} from the {source} dataset.")

    # Addresses
    addresses = record.get("ADDRESSES", [])
    if addresses:
        addr_strs = [a.get("ADDR_FULL", "") for a in addresses if a.get("ADDR_FULL")]
        if addr_strs:
            parts.append(f"Address: {'; '.join(addr_strs)}.")

    # Dates
    for date_entry in record.get("DATES", []):
        if date_entry.get("DATE_OF_BIRTH"):
            parts.append(f"Date of birth: {date_entry['DATE_OF_BIRTH']}.")
        if date_entry.get("REGISTRATION_DATE"):
            parts.append(f"Registration date: {date_entry['REGISTRATION_DATE']}.")

    # Top-level dates (Open Ownership)
    if record.get("REGISTRATION_DATE"):
        parts.append(f"Registration date: {record['REGISTRATION_DATE']}.")
    if record.get("dissolutionDate"):
        parts.append(f"Dissolution date: {record['dissolutionDate']}.")

    # Countries
    for country_entry in record.get("COUNTRIES", []):
        if country_entry.get("NATIONALITY"):
            parts.append(f"Nationality: {country_entry['NATIONALITY'].upper()}.")
        if country_entry.get("REGISTRATION_COUNTRY"):
            parts.append(f"Registered in: {country_entry['REGISTRATION_COUNTRY'].upper()}.")
    if record.get("REGISTRATION_COUNTRY"):
        parts.append(f"Registered in: {record['REGISTRATION_COUNTRY'].upper()}.")

    # Attributes (Open Ownership persons)
    for attr in record.get("ATTRIBUTES", []):
        if attr.get("NATIONALITY"):
            parts.append(f"Nationality: {attr['NATIONALITY'].upper()}.")

    # Risks (Open Sanctions)
    risks = record.get("RISKS", [])
    if risks:
        topics = [r.get("TOPIC", "") for r in risks if r.get("TOPIC")]
        if topics:
            parts.append(f"Risk flags: {', '.join(topics)}.")

    # Identifiers
    id_parts = []
    for id_entry in record.get("IDENTIFIERS", []):
        id_type = id_entry.get("OTHER_ID_TYPE") or id_entry.get("NATIONAL_ID_TYPE") or ""
        id_num = id_entry.get("OTHER_ID_NUMBER") or id_entry.get("NATIONAL_ID_NUMBER") or ""
        if id_type and id_num:
            id_parts.append(f"{id_type}: {id_num}")
        elif id_num:
            id_parts.append(id_num)
    if id_parts:
        parts.append(f"Identifiers: {', '.join(id_parts)}.")

    # Relationships
    rels = record.get("RELATIONSHIPS", [])
    pointer_rels = [r for r in rels if r.get("REL_POINTER_ROLE")]
    if pointer_rels:
        rel_strs = []
        for r in pointer_rels:
            role = r["REL_POINTER_ROLE"]
            domain = r.get("REL_POINTER_DOMAIN", "")
            key = r.get("REL_POINTER_KEY", "")
            from_date = r.get("REL_POINTER_FROM_DATE", "")
            thru_date = r.get("REL_POINTER_THRU_DATE", "")
            rel_str = f"{role} (ref: {domain}/{key})"
            if from_date:
                rel_str += f" from {from_date}"
            if thru_date:
                rel_str += f" to {thru_date}"
            rel_strs.append(rel_str)
        parts.append(f"Relationships: {'; '.join(rel_strs)}.")

    return " ".join(parts)


# Test with a few records
print("=== Sample Open Sanctions record ===")
print(record_to_text(sanctions_records[0]))
print()
print("=== Sample Open Ownership record (org) ===")
print(record_to_text(ownership_records[0]))
print()
# Find a person record in ownership
person_oo = next((r for r in ownership_records if r.get("RECORD_TYPE") == "PERSON"), None)
if person_oo:
    print("=== Sample Open Ownership record (person) ===")
    print(record_to_text(person_oo))

=== Sample Open Sanctions record ===
Abassin BADSHAH is a person from the OPEN-SANCTIONS dataset. Address: 31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL. Date of birth: 1985-05-12. Nationality: GB. Risk flags: corp.disqual. Identifiers: OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ. Relationships: Directorship (ref: OPEN-SANCTIONS/NK-SKAADAiqiZ78JsJjeg72Te); Directorship (ref: OPEN-SANCTIONS/NK-3p3mmVWmjwVtTfKchz4kNE).

=== Sample Open Ownership record (org) ===
GOLD WYNN UK HOLDINGS LIMITED is a organization from the OPEN-OWNERSHIP dataset. Address: C/O Fladgate Llp, 16 Great Queen Street, London, WC2B 5DG. Registration date: 2020-03-18. Registered in: GB. Identifiers: GB-COH: 12524623. Relationships: shareholding 75% 100% (ref: OOR/7584591804488095167) from 2020-03-18 to 2020-04-29; voting_rights 75% 100% (ref: OOR/7584591804488095167) from 2020-03-18 to 2020-04-29; appointment_of_board (ref: OOR/7584591804488095167) from 2020-03-18 to 2020-04-29.

=== Sample Open Ownership 

## Vectorize and Store in LanceDB

Embed all text descriptions using the same `all-MiniLM-L6-v2` model, then store them alongside metadata in a fresh LanceDB table.

In [5]:
print("Loading embedding model...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Embedding model loaded (dimension: {embedding_model.get_sentence_embedding_dimension()})")

# Combine all records and build text + metadata
all_records = sanctions_records + ownership_records
print(f"\nProcessing {len(all_records)} records...")

rows = []
seen = set()
for record in all_records:
    # Deduplicate by (DATA_SOURCE, RECORD_ID)
    key = (record.get("DATA_SOURCE", ""), record.get("RECORD_ID", ""))
    if key in seen:
        continue
    seen.add(key)

    text = record_to_text(record)
    name = get_name(record)
    rows.append({
        "record_id": record.get("RECORD_ID", ""),
        "data_source": record.get("DATA_SOURCE", ""),
        "record_type": record.get("RECORD_TYPE", ""),
        "name": name,
        "text": text,
    })

print(f"Unique records to vectorize: {len(rows)}")

# Batch encode all texts
texts = [r["text"] for r in rows]
embeddings = embedding_model.encode(texts, show_progress_bar=True)

for i, row in enumerate(rows):
    row["vector"] = embeddings[i].tolist()

print(f"Embeddings created: {len(embeddings)} x {embeddings.shape[1]}")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded (dimension: 384)

Processing 340 records...
Unique records to vectorize: 282


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Embeddings created: 282 x 384


In [6]:
# Store in LanceDB
db = lancedb.connect(LANCEDB_PATH)
table = db.create_table("raw_records", data=rows)

print(f"LanceDB table 'raw_records' created with {table.count_rows()} rows")
print(f"\nSample entries:")
sample = table.to_pandas().head(3)
print(sample[["record_id", "data_source", "record_type", "name"]].to_string())

LanceDB table 'raw_records' created with 282 rows

Sample entries:
                   record_id     data_source   record_type                     name
0  NK-25vyVFzt8vdJGgAXMRTwTJ  OPEN-SANCTIONS        PERSON          Abassin BADSHAH
1  NK-3p3mmVWmjwVtTfKchz4kNE  OPEN-SANCTIONS  ORGANIZATION            LMAR (GB) LTD
2  NK-auyPsLrBzRoxjCRWgjBvas  OPEN-SANCTIONS  ORGANIZATION  WANDLE HOLDINGS LIMITED


## Set Up LLM Clients

In [7]:
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Anthropic client ready:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("OpenAI client ready:", bool(os.getenv("OPENAI_API_KEY")))

Anthropic client ready: True
OpenAI client ready: True


## RAG Query Function

Vector search over raw records, then query either Anthropic (Claude) or OpenAI (GPT) to answer the question. The `provider` parameter controls which LLM backend is used.

In [9]:
SYSTEM_PROMPT = (
    "Answer questions about a corporate ownership and sanctions dataset. "
    "You have access to individual raw source records from Open Sanctions and "
    "Open Ownership. These records have NOT been entity-resolved — the same "
    "real-world entity may appear as separate records with slightly different "
    "names or details. You do not have relationship or graph data beyond what "
    "is mentioned in individual records."
)


def ask_raw_rag(question: str, provider: str = "anthropic", top_k: int = 10) -> None:
    """
    RAG over raw (non-entity-resolved) records.

    Parameters
    ----------
    question : str
        The user's question.
    provider : str
        LLM backend — "anthropic" (Claude) or "openai" (GPT).
    top_k : int
        Number of records to retrieve.
    """
    if provider not in ("anthropic", "openai"):
        raise ValueError(f"provider must be 'anthropic' or 'openai', got '{provider}'")

    print(f"\nQuestion: {question}")
    print(f"Provider: {provider}")
    print("=" * 70)

    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    print(f"Retrieved {len(results)} records")

    # Step 2: Build context
    context_parts = ["RAW SOURCE RECORDS:\n"]
    for r in results:
        context_parts.append(f"- [{r['data_source']}] {r['text']}")
        context_parts.append("")
    context = "\n".join(context_parts)

    # Step 3: Query LLM
    print("Querying LLM...")
    user_message = f"Context:\n{context}\n\nQuestion: {question}"

    if provider == "anthropic":
        response = anthropic_client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        answer = response.content[0].text
    else:
        response = openai_client.chat.completions.create(
            model="gpt-5.4-nano",
            max_tokens=2048,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
        )
        answer = response.choices[0].message.content

    print("\n" + "=" * 70)
    print("ANSWER")
    print("=" * 70)
    print(answer)
    print("=" * 70)

## Interactive Query Session

Ask questions against the raw (non-entity-resolved) data. Change `provider` below to `"openai"` to use GPT instead of Claude.

In [ ]:
provider = "anthropic"  # change to "openai" to use GPT-5.4 nano

print("RAG on Raw Data (No Entity Resolution) - Interactive Session")
print("=" * 70)
print("Ask questions about the corporate ownership and sanctions data.")
print("These are RAW records — no entity resolution has been applied.")
print(f"Current LLM provider: {provider}")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break

    if not question:
        continue

    try:
        ask_raw_rag(question, provider=provider)
        print()
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

RAG on Raw Data (No Entity Resolution) - Interactive Session
Ask questions about the corporate ownership and sanctions data.
These are RAW records — no entity resolution has been applied.
Current LLM provider: anthropic
Type 'quit' to exit.



Your question:  tell me about said kerimov



Question: tell me about said kerimov
Provider: anthropic
Retrieved 10 records
Querying LLM...

ANSWER
# Said Kerimov

Based on the available records, here's what we know about Said Kerimov:

## Basic Information
- **Date of Birth:** July 6, 1995
- **Nationality:** Russian (RU)
- **Wikidata ID:** Q44155839

## Addresses
Said Kerimov has been associated with multiple addresses:
- **Moscow, Russia:** Apt 270, Build. 31, Pyatnitskoe Shosse, 123430 Moscow
- **London, UK:** 
  - 8th Floor, 20 Farringdon Street, London, EC4A 4AB
  - 16 Berkeley Street, London, W1J 8DZ

## Risk Profile
- **Sanctioned individual** (appears in sanctions datasets)
- **Risk flags:** role.rca (likely "Relative or Close Associate"), sanction

## Family Connection
Said Kerimov has a documented family relationship with **Suleyman Abusaidovich Kerimov** (ref: Q447250), who is:
- A prominent Russian-French oligarch and politically exposed person (PEP)
- Born March 12, 1966
- Also sanctioned
- Based in Moscow with conne

Your question:  tell me about nugget capital


Traceback (most recent call last):
  File "/tmp/ipykernel_18819/234696214.py", line 21, in <module>
    ask_raw_rag(question, provider=provider)
  File "/tmp/ipykernel_18819/34865451.py", line 33, in ask_raw_rag
    results = table.search(question_embedding).limit(top_k).to_list()
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/lancedb/query.py", line 771, in to_list
    return self.to_arrow(timeout=timeout).to_pylist()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/lancedb/query.py", line 1286, in to_arrow
    return self.to_batches(timeout=timeout).read_all()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/lancedb/query.py", line 1343, in to_batches
    result_set = self._table._execute_query(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/lancedb/table.py", line 2871, in _execute_que


Question: tell me about nugget capital
Provider: anthropic
Error: lance error: Not found: workspace/lancedb_data/raw_records.lance/data/1111110001111010101111008996634ef78d4c6387c07e2e5f.lance, /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/lance-io-2.0.0/src/local.rs:139:31
